In [1]:
%load_ext autoreload
%autoreload 2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, SequentialSampler, RandomSampler

from einops import rearrange
from functools import partial

import matplotlib.pyplot as plt
import logging
import copy
import numpy as np

import math
import os, json
from typing import List,Tuple,Dict
import tqdm

import sys

sys.path.append(os.path.abspath('../ai4flow/acdm'))

from turbpred.data_transformations import Transforms
from turbpred.model_diffusion import DiffusionModel
from turbpred.params import DataParams
from turbpred.model import PredictionModel
from turbpred.logger import Logger
from turbpred.params import DataParams, TrainingParams, LossParams, ModelParamsEncoder, ModelParamsDecoder, ModelParamsLatent
from turbpred.turbulence_dataset import TurbulenceDataset
from turbpred.data_transformations import Transforms
from turbpred.loss import PredictionLoss
from turbpred.loss_history import LossHistory
from turbpred.trainer_diffusion import TrainerDiffusion, TesterDiffusion

/home/chunyang/miniconda3/envs/ACDM/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from turbpred.v2d_dataset import V2dDataset

# index for v2d dataset range: [54452, 99999]

trainSet = V2dDataset(name="V2D dataset",
                     dataDir="/home/chunyang/projects/DDPM4SCIENCE/data/v2d",
                     idx_range=[54452, 90888],
                     sequenceLength=[1, 1], rey=1000)

Loading data from 	 /home/chunyang/projects/DDPM4SCIENCE/data/v2d
Loading completed. Files loaded: 	36436
Loading completed. Files loaded: 	36436

Dataset info detail:
	 Sequence length of each sample: 1
	 Sequence skiping samples of: 1


In [4]:
# get mean and variance of the data distribution
trainSet[0]["data"].shape

(1, 4, 128, 64)

In [5]:
trainSet[0]["data"].shape

(1, 4, 128, 64)

In [6]:
from scipy.stats import describe
for i in range(3):
    print(describe((trainSet[0]["data"][:, i, :, :].flatten()+1e-6)))

DescribeResult(nobs=8192, minmax=(-7.5513483178999685, 1.6029567593207188), mean=0.9750936676512256, variance=0.11633328224036484, skewness=-14.868865162924282, kurtosis=295.98990539127914)
DescribeResult(nobs=8192, minmax=(-7.784103359505647, 2.4469030373707135), mean=-0.0077450674027093915, variance=0.10417468285942327, skewness=-6.759128849773102, kurtosis=133.48264934872844)
DescribeResult(nobs=8192, minmax=(-1.3057484142524751, 0.483062870644698), mean=-0.045006240630650035, variance=0.024753792878004995, skewness=-3.114627467971713, kurtosis=11.745236246768291)


In [7]:
u_list = []
v_list = []
p_list = []

for i in tqdm.tqdm(range(len(trainSet))):
    u = trainSet[i]["data"][0, 0, :, :]
    v = trainSet[i]["data"][0, 1, :, :]
    p = trainSet[i]["data"][0, 2, :, :]
    u_list.append(u)
    v_list.append(v)
    p_list.append(p)

  0%|          | 0/36436 [00:00<?, ?it/s]

  0%|          | 5/36436 [00:00<13:03, 46.52it/s]

  0%|          | 16/36436 [00:00<12:49, 47.34it/s]

  0%|          | 31/36436 [00:00<11:28, 52.86it/s]

  0%|          | 44/36436 [00:00<11:09, 54.35it/s]

...

100%|█████████▉| 36383/36436 [12:23<00:00, 60.02it/s]

100%|█████████▉| 36397/36436 [12:23<00:00, 60.10it/s]

100%|█████████▉| 36411/36436 [12:23<00:00, 55.09it/s]

100%|█████████▉| 36425/36436 [12:23<00:00, 61.08it/s]

100%|██████████| 36436/36436 [12:24<00:00, 48.97it/s]

In [ ]:
def find_mean_and_std(input):
    mean = np.mean(input)
    std = np.std(input)
    return mean, std

u_mean, u_std = find_mean_and_std(np.stack(u_list).flatten())
v_mean, v_std = find_mean_and_std(np.stack(v_list).flatten())
p_mean, p_std = find_mean_and_std(np.stack(p_list).flatten())

print(u_mean, u_std)
print(v_mean, v_std)
print(p_mean, p_std)

0.9749698 0.33951518
-0.005726286 0.3157814
-0.041875295 0.15982394


In [ ]:
from turbpred.v2d_dataset import V2dDataset

# index for v2d dataset range: [54452, 99999]

trainSet = V2dDataset(name="V2D dataset",
                     dataDir="/home/chunyang/projects/autoreg-pde-diffusion/data/v2d/",
                     idx_range=[54452, 99999],
                     sequenceLength=[1, 1], rey=1000)

u_list = []
v_list = []
p_list = []

for i in tqdm.tqdm(range(len(trainSet))):
    u = trainSet[i]["data"][0, 0, :, :]
    v = trainSet[i]["data"][0, 1, :, :]
    p = trainSet[i]["data"][0, 2, :, :]
    u_list.append(u)
    v_list.append(v)
    p_list.append(p)
    
u_mean, u_std = find_mean_and_std(np.stack(u_list).flatten())
v_mean, v_std = find_mean_and_std(np.stack(v_list).flatten())
p_mean, p_std = find_mean_and_std(np.stack(p_list).flatten())

print(u_mean, u_std)
print(v_mean, v_std)
print(p_mean, p_std)

Loading data from 	 /home/chunyang/projects/autoreg-pde-diffusion/data/v2d/
Loading completed. Files loaded: 	45547
Loading completed. Files loaded: 	45547

Dataset info detail:
	 Sequence length of each sample: 1
	 Sequence skiping samples of: 1


100%|██████████| 45547/45547 [01:55<00:00, 395.54it/s]


0.97507006 0.33952948
-0.005726456 0.31580338
-0.04188044 0.15982327


# Inspect mean and std. after transformation

In [ ]:
from turbpred.v2d_transformations import Transforms

trainSet = V2dDataset(name="V2D dataset",
                     dataDir="/home/chunyang/projects/DDPM4SCIENCE/data/v2d",
                     idx_range=[54452, 90888],
                     sequenceLength=[1, 1], rey=1000)

transTrain = Transforms()
trainSet.transform = transTrain


u_list = []
v_list = []
p_list = []

for i in tqdm.tqdm(range(len(trainSet))):
    u = trainSet[i]["data"][0, 0, :, :]
    v = trainSet[i]["data"][0, 1, :, :]
    p = trainSet[i]["data"][0, 2, :, :]
    u_list.append(u)
    v_list.append(v)
    p_list.append(p)
    
u_mean, u_std = find_mean_and_std(np.stack(u_list).flatten())
v_mean, v_std = find_mean_and_std(np.stack(v_list).flatten())
p_mean, p_std = find_mean_and_std(np.stack(p_list).flatten())

print(u_mean, u_std)
print(v_mean, v_std)
print(p_mean, p_std)

Loading data from 	 /home/chunyang/projects/autoreg-pde-diffusion/data/v2d/
Loading completed. Files loaded: 	36436
Loading completed. Files loaded: 	36436

Dataset info detail:
	 Sequence length of each sample: 1
	 Sequence skiping samples of: 1


100%|██████████| 36436/36436 [01:43<00:00, 352.43it/s]


8.898813e-05 1.0000212
-1.9819445e-05 1.0000001
-3.1959906e-05 1.0000237


In [ ]:
trainSet[0]

{'data': tensor([[[[ 0.0861,  0.0867,  0.0871,  ...,  0.0996,  0.0993,  0.0985],
           [ 0.0878,  0.0883,  0.0886,  ...,  0.1024,  0.1018,  0.1009],
           [ 0.0895,  0.0900,  0.0904,  ...,  0.1042,  0.1034,  0.1024],
           ...,
           [ 0.1127,  0.1136,  0.1144,  ...,  0.1146,  0.1147,  0.1134],
           [ 0.1131,  0.1152,  0.1175,  ...,  0.1030,  0.1036,  0.1024],
           [ 0.1129,  0.1151,  0.1176,  ...,  0.0979,  0.0993,  0.0990]],
 
          [[-0.0211, -0.0224, -0.0239,  ...,  0.0583,  0.0564,  0.0544],
           [-0.0271, -0.0286, -0.0308,  ...,  0.0597,  0.0576,  0.0555],
           [-0.0272, -0.0289, -0.0312,  ...,  0.0607,  0.0584,  0.0562],
           ...,
           [ 0.0045,  0.0050,  0.0056,  ...,  0.0359,  0.0346,  0.0341],
           [-0.0045, -0.0031, -0.0028,  ...,  0.0584,  0.0563,  0.0553],
           [-0.0079, -0.0068, -0.0056,  ...,  0.0702,  0.0668,  0.0639]],
 
          [[ 0.3830,  0.3835,  0.3845,  ...,  0.3671,  0.3622,  0.3610],
     